In [0]:
%sql
USE CATALOG adb_dream_analytics_prod;
CREATE SCHEMA IF NOT EXISTS gold MANAGED LOCATION 'abfss://gold@adlsgen2dreamprod.dfs.core.windows.net/dream_app/';
USE SCHEMA gold;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold.fact_trips (
    trip_id INT,
    driver_sk BIGINT,
    customer_sk BIGINT,
    pickup_location_sk BIGINT,
    dropoff_location_sk BIGINT,
    trip_start_time TIMESTAMP,
    trip_end_time TIMESTAMP,
    distance_km DECIMAL(18,2),
    fare_amount DECIMAL(18,2),
    payment_method STRING,
    trip_status STRING,
    _processed_at TIMESTAMP
) USING DELTA;

CREATE OR REPLACE TEMPORARY VIEW v_fact_trips AS
SELECT
    t.trip_id,
    d.driver_sk,
    c.customer_sk,
    pl.location_id AS pickup_location_sk ,
    dl.location_id AS dropoff_location_sk,
    t.trip_start_time,
    t.trip_end_time,
    t.distance_km,
    t.fare_amount,
    t.payment_method,
    t.trip_status,
    current_timestamp() AS _processed_at
FROM adb_dream_analytics_prod.silver.trips t
LEFT JOIN gold.dim_drivers d ON t.driver_id = d.driver_id
    AND t.trip_start_time BETWEEN d.start_date AND nvl(d.end_date, '9999-12-31')
LEFT JOIN gold.dim_customers c ON t.customer_id = c.customer_id
    AND t.trip_start_time BETWEEN c.start_date AND nvl(c.end_date, '9999-12-31')
LEFT JOIN gold.dim_locations pl ON t.start_location = pl.city
LEFT JOIN gold.dim_locations dl ON t.end_location = dl.city;

TRUNCATE TABLE fact_trips;
INSERT INTO fact_trips SELECT * FROM v_fact_trips;
OPTIMIZE fact_trips ZORDER BY (trip_start_time);